# **Processamento Inicial e Análise Exploratória de Dados da API do The Cancer Genome Atlas (TCGA)**
Desenvolvida pela Genomic Data Commons (GDC)

# Importação de Bibliotecas

In [1]:
import json
import os
import re

import pandas as pd
import requests

# Constantes

In [2]:
# URL HTML base da API TCGA / GDC
TCGA_API_ENDPOINT = 'https://api.gdc.cancer.gov'

# Endpoint de casos relacionados aos projetos
CASES_ENDPOINT = f'{TCGA_API_ENDPOINT}/cases'

# Endpoint para download dos arquivos
DATA_ENDPOINT = f'{TCGA_API_ENDPOINT}/data'

# Endpoint de arquivos relacionados aos casos
FILES_ENDPOINT = f'{TCGA_API_ENDPOINT}/files'

# Endpoint de projetos relacionados aos programas
PROJECTS_ENDPOINT = f'{TCGA_API_ENDPOINT}/projects'

# Endpoint para verificar status e versão da API
STATUS_ENDPOINT = f'{TCGA_API_ENDPOINT}/status'

# Caminho da pasta de dados externos
EXTERNAL_DATA_PATH = '../../data/external'

# Status da API

In [3]:
# Requisita o status e a versão da API ao endpoint
response = requests.get(STATUS_ENDPOINT)

# Printa o conteúdo da resposta
print(response.content.decode('utf-8'))

{"commit":"3ed4235efe2f2d9c9969e1ce790e6b34a5c60c9d","data_release":"Data Release 41.0 - August 28, 2024","status":"OK","tag":"7.4.0","version":1}



# Processamento de Dados

## Total de Projetos

In [4]:
# Requisita as informações contidas no endpoint
response = requests.get(f'{PROJECTS_ENDPOINT}/_mapping')

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))

# Armazena os campos contidos no endpoint de projetos
fields_projects = json.dumps(json_response['fields'])[1:-2]
fields_projects = fields_projects.replace('"', '')
fields_projects = fields_projects.replace(' ', '')

In [5]:
# Requisita os objetos do endpoint
response = requests.post(
    url=PROJECTS_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json={'fields': 'name'}
)

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))

# Armazena total de projetos contidos no TCGA
total_projects = json.dumps(json_response['data']['pagination']['total'])
print(f'Total de projetos no TCGA: {total_projects}')

Total de projetos no TCGA: 86


In [6]:
# Parâmetros para a requisição ao endpoint
params = {
    'fields': fields_projects,
    'size': total_projects
}

# Requisita todos os objetos do endpoint
response = requests.post(
    url=PROJECTS_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json=params
)

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))['data']['hits']

# Transforma os valores booleanos do JSON em strings
for index in range(len(json_response)):
    obj = json_response[index]
    for key in obj:
        if isinstance(obj[key], bool): 
            obj[key] = str(obj[key])

# Converte o JSON para um DataFrame pandas
df_all_projects = pd.json_normalize(json_response)

# Printa o DataFrame parcialmente
df_all_projects.head(3)

,id,primary_site,dbgap_accession_number,project_id,disease_type,name,releasable,state,released,summary.file_count,summary.data_categories,summary.experimental_strategies,summary.case_count,summary.file_size,program.dbgap_accession_number,program.program_id,program.name
0,TARGET-AML,"[Unknown, Hematopoietic and reticuloendothelia...",phs000465,TARGET-AML,"[Not Applicable, Myeloid Leukemias]",Acute Myeloid Leukemia,True,open,True,51339,"[{'file_count': 12095, 'case_count': 806, 'dat...","[{'file_count': 28629, 'case_count': 2280, 'ex...",2492,104391798872383,phs000218,f1c391e9-8488-55a8-b777-302e786ea11d,TARGET
1,MATCH-Z1I,"[Bronchus and lung, Gallbladder, Pancreas, Unk...",phs002058,MATCH-Z1I,"[Squamous Cell Neoplasms, Epithelial Neoplasms...",Genomic Characterization CS-MATCH-0007 Arm Z1I,False,open,True,660,"[{'file_count': 327, 'case_count': 24, 'data_c...","[{'file_count': 374, 'case_count': 24, 'experi...",26,2396549656405,None,893b0820-b50b-5c75-85f5-3b028e251811,MATCH
2,HCMI-CMDC,"[Breast, Rectum, Nasal cavity and middle ear, ...",None,HCMI-CMDC,"[Cystic, Mucinous and Serous Neoplasms, Ductal...",NCI Cancer Model Development for the Human Can...,True,open,True,20761,"[{'file_count': 8450, 'case_count': 278, 'data...","[{'file_count': 8096, 'case_count': 276, 'expe...",278,130722797290059,phs001486,a5448c11-d46a-56aa-a5e1-5c1aa06404df,HCMI


## Projetos Lançados com miRNA-Seq e RNA-Seq

In [7]:
# Filtra os projetos lançados pelo TCGA e remove as colunas desnecessárias
df_projects = df_all_projects \
    .query('released == "True"') \
    .drop(
        columns=[
            'dbgap_accession_number',
            'id',
            'releasable',
            'released',
            'program.dbgap_accession_number',
            'state',
            'summary.data_categories',
            'summary.file_count',
            'summary.file_size'
        ]
    )

# Inicializa listas para a contagem de casos com miRNA-Seq e RNA-Seq nos projetos
miRNA_Seq_case_count = [0] * df_projects.shape[0]
RNA_Seq_case_count = [0] * df_projects.shape[0]

# Preenche listas com a contagem de casos com miRNA-Seq e RNA-Seq em cada projeto
for index in range(df_projects.shape[0]):
    for info in df_projects['summary.experimental_strategies'][index]:
        if info['experimental_strategy'] in ['miRNA-Seq', 'RNA-Seq']:
            if info['experimental_strategy'] == 'miRNA-Seq':
                miRNA_Seq_case_count[index] = info['case_count']
            else:
                RNA_Seq_case_count[index] = info['case_count']

# Adiciona a contagem de casos de miRNA-Seq e RNA-Seq ao DataFrame
df_projects['miRNA-Seq_case_count'] = miRNA_Seq_case_count
df_projects['RNA-Seq_case_count'] = RNA_Seq_case_count

# Filtra os projetos com miRNA-Seq e RNA-Seq como estratégia experimental
df_projects = df_projects \
    .query('(`miRNA-Seq_case_count` > 0) & (`RNA-Seq_case_count` > 0)') \
    .drop(columns='summary.experimental_strategies') \
    .reset_index(drop=True)

# Reorganiza e renomeia as colunas do DataFrame
columns = [
    'project_id',
    'name',
    'primary_site',
    'disease_type',
    'summary.case_count',
    'miRNA-Seq_case_count',
    'RNA-Seq_case_count',
    'program.program_id',
    'program.name'
]
df_projects = df_projects[columns] \
    .rename(
        columns={
            'name': 'project_name',
            'summary.case_count': 'case_count',
            'program.program_id': 'program_id',
            'program.name': 'program_name'
        }
    )

In [8]:
# Printa o DataFrame
pd.set_option('display.max_colwidth', 100)
df_projects

,project_id,project_name,primary_site,disease_type,case_count,miRNA-Seq_case_count,RNA-Seq_case_count,program_id,program_name
0,TARGET-AML,Acute Myeloid Leukemia,"[Unknown, Hematopoietic and reticuloendothelial systems]","[Not Applicable, Myeloid Leukemias]",2492,1836,2280,f1c391e9-8488-55a8-b777-302e786ea11d,TARGET
1,HCMI-CMDC,NCI Cancer Model Development for the Human Cancer Model Initiative,"[Breast, Rectum, Nasal cavity and middle ear, Bronchus and lung, Other and unspecified parts of ...","[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",278,271,277,a5448c11-d46a-56aa-a5e1-5c1aa06404df,HCMI
2,TCGA-PCPG,Pheochromocytoma and Paraganglioma,"[Other endocrine glands and related structures, Heart, mediastinum, and pleura, Connective, subc...",[Paragangliomas and Glomus Tumors],179,179,179,b80aa962-9650-5110-b3eb-bd087da808db,TCGA
3,TCGA-THYM,Thymoma,"[Heart, mediastinum, and pleura, Thymus]",[Thymic Epithelial Neoplasms],124,124,120,b80aa962-9650-5110-b3eb-bd087da808db,TCGA
4,TCGA-PAAD,Pancreatic Adenocarcinoma,[Pancreas],"[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",185,178,178,b80aa962-9650-5110-b3eb-bd087da808db,TCGA
5,TCGA-STAD,Stomach Adenocarcinoma,[Stomach],"[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas]",443,436,415,b80aa962-9650-5110-b3eb-bd087da808db,TCGA
6,TCGA-TGCT,Testicular Germ Cell Tumors,[Testis],[Germ Cell Neoplasms],263,150,150,b80aa962-9650-5110-b3eb-bd087da808db,TCGA
7,TCGA-SARC,Sarcoma,"[Bones, joints and articular cartilage of limbs, Uterus, NOS, Corpus uteri, Other and unspecifie...","[Lipomatous Neoplasms, Synovial-like Neoplasms, Fibromatous Neoplasms, Nerve Sheath Tumors, Myom...",261,259,259,b80aa962-9650-5110-b3eb-bd087da808db,TCGA
8,TCGA-PRAD,Prostate Adenocarcinoma,[Prostate gland],"[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas, Ductal and Lobular Neoplasms]",500,494,497,b80aa962-9650-5110-b3eb-bd087da808db,TCGA
9,TCGA-READ,Rectum Adenocarcinoma,"[Rectum, Rectosigmoid junction, Unknown, Colon, Connective, subcutaneous and other soft tissues]","[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas]",172,161,167,b80aa962-9650-5110-b3eb-bd087da808db,TCGA


## Casos com miRNA-Seq e RNA-Seq

In [9]:
# Campos de interesse para a requisição ao endpoint
fields_cases = [
    'disease_type',
    'files.access',
    'files.created_datetime',
    'files.data_category',
    'files.data_format',
    'files.data_type',
    'files.file_id',
    'files.experimental_strategy',
    'primary_site',
    'project.project_id',
]
fields_cases = ','.join(fields_cases)

# Inicializa o DataFrame de casos e arquivos
df_cases_and_files = pd.DataFrame()

# Busca pelos casos com miRNA-Seq e RNA-Seq de cada projeto de interesse
for index in range(df_projects.shape[0]):
    # Filtros empregados na requisição ao endpoint
    filters = {
        'op': 'and',
        'content': [
            {
                'op': '=',
                'content': {
                    'field': 'project.project_id',
                    'value': df_projects['project_id'][index]
                }
            },
            {
                'op': 'in',
                'content': {
                    'field': 'files.experimental_strategy',
                    'value': ['miRNA-Seq', 'RNA-Seq']
                }
            }
        ]
    }

    # Parâmetros para a requisição ao endpoint
    params = {
        'fields': fields_cases,
        'filters': filters,
        'size': str(df_projects['case_count'][index])
    }

    # Requisita todos os objetos do projeto no endpoint
    response = requests.post(
        url=CASES_ENDPOINT,
        headers={'Content-Type': 'application/json'},
        json=params
    )

    # Converte o conteúdo da resposta para JSON
    json_response = json.loads(response.content.decode('utf-8'))

    # Converte o JSON para um DataFrame pandas
    df_project_cases = pd.json_normalize(json_response['data']['hits'])

    # Concatena os casos deste projeto com os restantes
    if df_cases_and_files.empty == False:
        df_cases_and_files = pd.concat(
            [df_cases_and_files, df_project_cases], ignore_index=True
        )
    else:
        df_cases_and_files = df_project_cases.copy()

# Renomeia algumas colunas do DataFrame
df_cases_and_files = df_cases_and_files.rename(
    columns={'id': 'case_id', 'project.project_id': 'project_id'}
)

In [10]:
# Printa o DataFrame de casos e arquivos
df_cases_and_files

,case_id,primary_site,disease_type,files,project_id
0,58771370-5082-485e-ac68-13edfbd9ef0c,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'XLSX', 'access': 'open', 'file_id': '52fd584b-9ca3-4f7e-bdd5-fd9dce3d630b', 'd...",TARGET-AML
1,28da5b52-29ed-4817-a678-9206c5164da0,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'XLSX', 'access': 'open', 'file_id': '52fd584b-9ca3-4f7e-bdd5-fd9dce3d630b', 'd...",TARGET-AML
2,28dae019-4d93-42bf-9bb3-8e6e21bc1d29,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'TSV', 'access': 'controlled', 'file_id': '46894ff9-3571-4878-8134-d0ad10a3fd89...",TARGET-AML
3,28f7e68c-e0ab-4fb0-b835-2e506cb0012d,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'BAM', 'access': 'controlled', 'file_id': '87a249be-2899-4294-9fea-4a7578971f2d...",TARGET-AML
4,28ffdfb9-f051-4424-ad35-e6633e6bf42e,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'XLSX', 'access': 'open', 'file_id': '52fd584b-9ca3-4f7e-bdd5-fd9dce3d630b', 'd...",TARGET-AML
...,...,...,...,...,...
16606,a798a8cc-4e72-4a8c-9e20-74a14deafd12,Corpus uteri,Adenomas and Adenocarcinomas,"[{'data_format': 'BCR Biotab', 'access': 'open', 'file_id': '3eefa637-ec64-42fa-8d0d-c740de10666...",TCGA-UCEC
16607,fe2e89f7-8f4d-420a-a551-4877cf0fd1d3,Corpus uteri,Adenomas and Adenocarcinomas,"[{'data_format': 'SVS', 'access': 'open', 'file_id': '0d29b22e-3f4f-462a-bc73-c93237cb3867', 'da...",TCGA-UCEC
16608,db3a4986-55d5-4ecc-be73-59725dce3c33,Corpus uteri,Adenomas and Adenocarcinomas,"[{'data_format': 'MAF', 'access': 'controlled', 'file_id': 'eb6f7308-a02d-414c-b2c0-7c86a7651df5...",TCGA-UCEC
16609,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8,Corpus uteri,"Cystic, Mucinous and Serous Neoplasms","[{'data_format': 'BAM', 'access': 'controlled', 'file_id': 'b0c765b4-4804-451c-812a-9cf2e86b570f...",TCGA-UCEC


## Arquivos com miRNA-Seq e RNA-Seq

In [11]:
# Explode as listas com dicionário sobre os arquivos dos casos
df_cases = df_cases_and_files.explode('files')

# Filtra os arquivos dos casos relacionados a miRNA-Seq e RNA-Seq
key = 'experimental_strategy'
df_cases = df_cases[
    df_cases['files'].apply(
        lambda x: (
            key in x and (x[key] == 'miRNA-Seq' or x[key] == 'RNA-Seq')
        )
    )
]

# Explode os dicionários com as informações sobre os arquivos de interesse
df_files = pd.json_normalize(df_cases['files'])

# Remove a coluna referente aos arquivos e redefine os índices do DataFrame
df_cases = df_cases.drop(columns='files').reset_index(drop=True)

# Concatena os DataFrames e define o DataFrame de arquivos
df_files = pd.concat([df_cases, df_files], axis='columns')
df_files = df_files.drop(columns=['primary_site', 'disease_type', 'project_id'])
df_files = df_files[[
    'file_id',
    'access',
    'created_datetime',
    'data_format',
    'data_type',
    'data_category',
    'experimental_strategy',
    'case_id'
]]

# Define o DataFrame de casos
df_cases = df_cases \
    .drop_duplicates() \
    .reset_index(drop=True)

In [12]:
# Printa o DataFrame de casos
df_cases

,case_id,primary_site,disease_type,project_id
0,58771370-5082-485e-ac68-13edfbd9ef0c,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
1,28da5b52-29ed-4817-a678-9206c5164da0,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
2,28dae019-4d93-42bf-9bb3-8e6e21bc1d29,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
3,28f7e68c-e0ab-4fb0-b835-2e506cb0012d,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
4,28ffdfb9-f051-4424-ad35-e6633e6bf42e,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
...,...,...,...,...
16606,a798a8cc-4e72-4a8c-9e20-74a14deafd12,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16607,fe2e89f7-8f4d-420a-a551-4877cf0fd1d3,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16608,db3a4986-55d5-4ecc-be73-59725dce3c33,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16609,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8,Corpus uteri,"Cystic, Mucinous and Serous Neoplasms",TCGA-UCEC


In [13]:
# Printa o DataFrame de arquivos
df_files

,file_id,access,created_datetime,data_format,data_type,data_category,experimental_strategy,case_id
0,7e37b56e-4a47-4e14-aa0c-2317e31f3e53,controlled,2022-07-08T10:28:50.128486-05:00,TSV,Transcript Fusion,Structural Variation,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
1,2b213562-4c34-4976-9563-ef2e581eb993,controlled,2022-07-07T19:08:01.345658-05:00,BAM,Aligned Reads,Sequencing Reads,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
2,4188fa7a-7b11-47ae-ad63-701f5a2ffecb,controlled,2022-07-08T10:11:13.883479-05:00,BEDPE,Transcript Fusion,Structural Variation,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
3,dd250c15-0be7-4243-b147-d4c872587844,controlled,2022-07-08T09:54:36.696020-05:00,BEDPE,Transcript Fusion,Structural Variation,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
4,619cea2a-d035-4979-ad83-b7863eefc337,controlled,2022-07-07T20:43:43.696288-05:00,BAM,Aligned Reads,Sequencing Reads,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
...,...,...,...,...,...,...,...,...
229791,d78ca382-fc44-48fb-b40e-3c56bab0ba99,open,2018-03-20T10:18:50.374798-05:00,TXT,miRNA Expression Quantification,Transcriptome Profiling,miRNA-Seq,43476c88-86bc-4a65-9327-f0d24a41c971
229792,af52e052-4b18-41f6-8647-8370405c4b00,controlled,2021-12-13T21:45:40.941248-06:00,TSV,Splice Junction Quantification,Transcriptome Profiling,RNA-Seq,43476c88-86bc-4a65-9327-f0d24a41c971
229793,f491ad0e-78db-4ed0-9d80-9d26bb0f736d,controlled,2018-03-20T10:18:50.374798-05:00,BAM,Aligned Reads,Sequencing Reads,miRNA-Seq,43476c88-86bc-4a65-9327-f0d24a41c971
229794,36cd1c9b-ab47-4a09-96d6-848cd59c08dd,open,2021-12-13T21:28:20.663448-06:00,TSV,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,43476c88-86bc-4a65-9327-f0d24a41c971


# Análise Exploratória de Dados

## Arquivos

In [14]:
# Colunas usadas na agregação
columns = [
    'access',
    'experimental_strategy',
    'data_category',
    'data_type',
    'data_format'
]

# Agrega os dados sobre os arquivos e faz a contagem
df_files_agg = df_files \
    .groupby(columns) \
    .agg(count=pd.NamedAgg(column='file_id', aggfunc='count'))

# Faz a contagem normalizada dos dados dos arquivos
df_files_agg['%count'] = df_files_agg['count'] / df_files_agg['count'].sum()
df_files_agg['%count'] = (df_files_agg['%count'] * 100).round(2)

# Printa o resultado
df_files_agg

count  \
access     experimental_strategy data_category           data_type                         data_format          
controlled RNA-Seq               Sequencing Reads        Aligned Reads                     BAM          59324   
                                 Structural Variation    Transcript Fusion                 BEDPE        39118   
                                                                                           TSV          36983   
                                 Transcriptome Profiling Splice Junction Quantification    TSV          20202   
           miRNA-Seq             Sequencing Reads        Aligned Reads                     BAM          17989   
open       RNA-Seq               Transcriptome Profiling Gene Expression Quantification    TSV          20202   
           miRNA-Seq             Transcriptome Profiling Isoform Expression Quantification TSV           6328   
                                                                                           TXT          11661   
                                                         miRNA Expression Quantification   TSV           6328   
                                                                                           TXT          11661   

                                                                                                        %count  
access     experimental_strategy data_category           data_type                         data_format          
controlled RNA-Seq               Sequencing Reads        Aligned Reads                     BAM           25.82  
                                 Structural Variation    Transcript Fusion                 BEDPE         17.02  
                                                                                           TSV           16.09  
                                 Transcriptome Profiling Splice Junction Quantification    TSV            8.79  
           miRNA-Seq             Sequencing Reads        Aligned Reads                     BAM            7.83  
open       RNA-Seq               Transcriptome Profiling Gene Expression Quantification    TSV            8.79  
           miRNA-Seq             Transcriptome Profiling Isoform Expression Quantification TSV            2.75  
                                                                                           TXT            5.07  
                                                         miRNA Expression Quantification   TSV            2.75  
                                                                                           TXT            5.07

## Casos

In [15]:
# Define a condição de filtragem
condition = '(data_category == "Transcriptome Profiling") and (access == "open")'

# Filtra os arquivos e recupera as informações relativas aos casos
df_files_and_cases = df_files \
    .query(condition) \
    .reset_index(drop=True) \
    .merge(right=df_cases, on='case_id', how='inner')

### Origens Primárias por Caso

In [16]:
# Agrega os casos de acordo com suas origens primárias e faz a contagem
df_files_and_cases \
    .groupby(['case_id', 'primary_site']) \
    .agg(distinct_sites=pd.NamedAgg(column='primary_site', aggfunc='nunique')) \
    .sort_values(by='distinct_sites', ascending=False)

,,distinct_sites
case_id,primary_site,
0004d251-3f70-4395-b175-c94c2f5b1b81,Liver and intrahepatic bile ducts,1
ab45050e-2965-4d33-bff4-cd7a067ddd3d,Cervix uteri,1
aac5502e-147d-4ff9-9876-d9bffacf3dd0,Testis,1
aad167ab-4772-4616-afb3-265e00d4a3f0,Bladder,1
aad65d8f-47ba-5ad5-925e-5050b691f319,Hematopoietic and reticuloendothelial systems,1
...,...,...
54bacd1c-a6c0-5c28-9140-750e67395d87,Kidney,1
54c04bcc-ebd5-461e-bbd5-b4d791b00c95,Bladder,1
54c9850c-9b67-42ea-a04b-16a88f786dfa,Kidney,1


### Tipos de Doença por Caso

In [17]:
# Agrega os casos de acordo com seus tipos de doença e faz a contagem
df_files_and_cases \
    .groupby(['case_id', 'disease_type']) \
    .agg(distinct_diseases=pd.NamedAgg(column='disease_type', aggfunc='nunique')) \
    .sort_values(by='distinct_diseases', ascending=False)

,,distinct_diseases
case_id,disease_type,
0004d251-3f70-4395-b175-c94c2f5b1b81,Adenomas and Adenocarcinomas,1
ab45050e-2965-4d33-bff4-cd7a067ddd3d,Squamous Cell Neoplasms,1
aac5502e-147d-4ff9-9876-d9bffacf3dd0,Germ Cell Neoplasms,1
aad167ab-4772-4616-afb3-265e00d4a3f0,Transitional Cell Papillomas and Carcinomas,1
aad65d8f-47ba-5ad5-925e-5050b691f319,Myeloid Leukemias,1
...,...,...
54c04bcc-ebd5-461e-bbd5-b4d791b00c95,Transitional Cell Papillomas and Carcinomas,1
54c9850c-9b67-42ea-a04b-16a88f786dfa,Adenomas and Adenocarcinomas,1
54cf3bfe-51c0-4cca-9054-bf76b05a9371,"Cystic, Mucinous and Serous Neoplasms",1


### Arquivos por Caso

In [18]:
# Agrega os casos de acordo com os tipos de dados de seus arquivos e faz a contagem
df_files_and_cases \
    .groupby(['case_id', 'data_type']) \
    .agg(distinct_files=pd.NamedAgg(column='file_id', aggfunc='nunique')) \
    .sort_values(by='distinct_files', ascending=False)

distinct_files
case_id                              data_type                                        
842402de-519e-4588-ad49-19df18db899b Gene Expression Quantification                  8
3a4afbe7-60d6-4a0d-8166-56a04ff127b0 Gene Expression Quantification                  8
9ff6d022-6e23-4f44-a480-1b61929e6ee3 miRNA Expression Quantification                 6
                                     Isoform Expression Quantification               6
                                     Gene Expression Quantification                  6
...                                                                                ...
5c266025-e590-457c-af86-4c2dc9797267 Gene Expression Quantification                  1
                                     Isoform Expression Quantification               1
                                     miRNA Expression Quantification                 1
5c2b9799-4caa-407e-88fb-c198fec0bc83 Gene Expression Quantification                  1
fffdb1d9-58d1-425c-ac12-1e1e5f443bf7 miRNA Expression Quantification                 1

[47171 rows x 1 columns]

In [19]:
# Calcula a quantidade de casos com mais de um arquivo miRNA-Seq ou RNA-Seq associado
df_files_and_cases \
    .groupby(['case_id', 'data_type']) \
    .agg(distinct_files=pd.NamedAgg(column='file_id', aggfunc='nunique')) \
    .query('distinct_files > 1') \
    .sort_values(by='distinct_files', ascending=False) \
    .reset_index() \
    .nunique()

case_id           3322
data_type            3
distinct_files       6
dtype: int64

In [20]:
# Printa os registros de um caso específico
df_files_and_cases \
    .query('case_id == "9ff6d022-6e23-4f44-a480-1b61929e6ee3"') \
    .sort_values(by=['data_type', 'created_datetime']) \
    .reset_index(drop=True)

,file_id,access,created_datetime,data_format,data_type,data_category,experimental_strategy,case_id,primary_site,disease_type,project_id
0,b96247db-6f2a-4d26-9ac4-142e1c079e0e,open,2022-01-06T09:46:53.769541-06:00,TSV,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
1,93c8678d-7afd-4be2-92cb-c39ad7701b43,open,2022-01-06T09:46:56.688571-06:00,TSV,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
2,d12e199e-8471-4f60-8da2-2b479db61ab4,open,2022-01-06T09:47:32.472999-06:00,TSV,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
3,167073fa-9e38-4f8d-af1f-301ed3a8b5f7,open,2022-01-06T09:47:37.123345-06:00,TSV,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
4,c54a604e-9379-46f1-938c-e2d09f8538d8,open,2022-01-06T09:47:57.280420-06:00,TSV,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
5,ceb4dd9b-6f10-4358-8312-3b126952d3cc,open,2022-01-06T09:55:35.726930-06:00,TSV,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
6,0cf6cded-942f-4141-a4a5-35afb7082f37,open,2019-10-10T11:23:13.777284-05:00,TSV,Isoform Expression Quantification,Transcriptome Profiling,miRNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
7,d39cc122-925b-4292-9fe1-cab1d031bbd7,open,2019-12-13T08:15:27.709025-06:00,TSV,Isoform Expression Quantification,Transcriptome Profiling,miRNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
8,0afd73ac-8fbc-418f-9d58-b1da06da4c98,open,2020-10-16T17:02:06.560539-05:00,TSV,Isoform Expression Quantification,Transcriptome Profiling,miRNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
9,c22a38cc-064b-46b8-a576-2fcbcfec7ceb,open,2020-10-16T17:06:28.600735-05:00,TSV,Isoform Expression Quantification,Transcriptome Profiling,miRNA-Seq,9ff6d022-6e23-4f44-a480-1b61929e6ee3,Kidney,Adenomas and Adenocarcinomas,CPTAC-3


### Arquivos por Origem Primária

In [21]:
# Agrega os arquivos de acordo com a origem primária e faz a contagem
df_site_agg = df_files_and_cases \
    .groupby('primary_site') \
    .agg(count=pd.NamedAgg(column='file_id', aggfunc='count')) \
    .sort_values(by='count', ascending=False) \
    .reset_index()

# Printa o resultado
df_site_agg

,primary_site,count
0,Hematopoietic and reticuloendothelial systems,9707
1,Kidney,5468
2,Bronchus and lung,5433
3,Thyroid gland,4630
4,Breast,4022
5,Brain,2767
6,Colon,2033
7,Corpus uteri,1734
8,Prostate gland,1656
9,Ovary,1653


# Download de Arquivos

## Gene Expression Quantification

In [22]:
# UUID do arquivo
file_id = 'b96247db-6f2a-4d26-9ac4-142e1c079e0e'

# Requisita o download do arquivo ao endpoint 
response = requests.get(
    url=f'{DATA_ENDPOINT}/{file_id}', 
    headers={'Content-Type': 'application/json'}
)

# Obtém o nome do arquivo por meio da resposta do endpoint
response_head_cd = response.headers['Content-Disposition']
file_name = re.findall('filename=(.+)', response_head_cd)[0]

# Armazena o arquivo na pasta de dados externos
file_path = os.path.join(EXTERNAL_DATA_PATH, file_name)
with open(file_path, 'wb') as output_file:
    output_file.write(response.content)

## miRNA Expression Quantification

In [23]:
# UUID do arquivo
file_id = '0cf6cded-942f-4141-a4a5-35afb7082f37'

# Requisita o download do arquivo ao endpoint 
response = requests.get(
    url=f'{DATA_ENDPOINT}/{file_id}', 
    headers={'Content-Type': 'application/json'}
)

# Obtém o nome do arquivo por meio da resposta do endpoint
response_head_cd = response.headers['Content-Disposition']
file_name = re.findall('filename=(.+)', response_head_cd)[0]

# Armazena o arquivo na pasta de dados externos
file_path = os.path.join(EXTERNAL_DATA_PATH, file_name)
with open(file_path, 'wb') as output_file:
    output_file.write(response.content)